In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model

class CompatBatchNormalization(tf.keras.layers.BatchNormalization):
    def __init__(self, *args, renorm=False, renorm_clipping=None, renorm_momentum=0.99, **kwargs):
        super().__init__(*args, **kwargs)

MODEL_PATH = "../models/efficientnet_model.h5"
model = load_model(
    MODEL_PATH,
    compile=False,
    custom_objects={"BatchNormalization": CompatBatchNormalization}
)

print("Loaded model from:", MODEL_PATH)
print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)


In [30]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import pandas as pd

df = pd.read_csv("../data/raw/metadata.csv")

label_map = {
    'mel': 'melanoma',
    'nv': 'nevus',
    'bcc': 'basal cell carcinoma',
    'akiec': 'actinic keratosis',
    'bkl': 'benign keratosis',
    'df': 'dermatofibroma',
    'vasc': 'vascular lesion'
}

df['label'] = df['dx'].map(label_map)

df['image_path'] = df['image_id'].apply(
    lambda x: "../data/raw/images/" + x + ".jpg"
)

from sklearn.model_selection import train_test_split

_, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
_, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_dataframe(
    test_df,
    x_col='image_path',
    y_col='label',
    target_size=(224,224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

Found 1503 validated image filenames belonging to 7 classes.


In [31]:
import numpy as np

y_pred = model.predict(test_generator)
y_pred_classes = np.argmax(y_pred, axis=1)

y_true = test_generator.classes

47/47 ━━━━━━━━━━━━━━━━━━━━ 30s 635ms/step


In [32]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred_classes))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        49
           1       0.00      0.00      0.00        77
           2       0.00      0.00      0.00       165
           3       0.00      0.00      0.00        17
           4       0.00      0.00      0.00       167
           5       0.67      1.00      0.80      1006
           6       0.00      0.00      0.00        22

    accuracy                           0.67      1503
   macro avg       0.10      0.14      0.11      1503
weighted avg       0.45      0.67      0.54      1503



c:\Users\hadee\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hadee\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\hadee\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave